In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader



In [11]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='data', train=True,
                             download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False,
                             download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)


In [12]:
class MnistNN(nn.Module):
    def __init__(self):
        super(MnistNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layer1  = nn.Linear(784, 128)
        self.layer2  = nn.Linear(128, 64)
        self.layer3  = nn.Linear(64, 10)
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.layer3(x)
        return x

model = MnistNN()
print(model)

MnistNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

losses = []
for epoch in range(10):
    running_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss   = criterion(output, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")



Epoch 1 | Loss: 0.4021
Epoch 2 | Loss: 0.1868
Epoch 3 | Loss: 0.1379
Epoch 4 | Loss: 0.1079
Epoch 5 | Loss: 0.0932
Epoch 6 | Loss: 0.0842


In [ ]:
correct = 0
total   = 0
with torch.no_grad():
    for images, labels in test_loader:
        output     = model(images)
        _, predicted = torch.max(output, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {correct/total:.4f}")



In [ ]:
import matplotlib.pyplot as plt

examples = iter(test_loader)
images, labels = next(examples)

with torch.no_grad():
    output = model(images)
    _, predicted = torch.max(output, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'True:{labels[i]} Pred:{predicted[i]}')
    ax.axis('off')
plt.show()
